In [1]:
import requests
from google.colab import userdata

API_KEY = userdata.get("API_KEY_OPENROUTER")

if not API_KEY:
    raise ValueError("Secret OPENROUTER_API_KEY belum ditemukan di Colab Secrets.")

BASE_URL = "https://openrouter.ai/api/v1"

headers = {
    "Authorization": f"Bearer {API_KEY}",
}

def money(x):
    if x is None:
        return "Tidak ada / unlimited / tidak disetel"
    return f"${float(x):,.6f}"

def get_openrouter_key_info():
    url = f"{BASE_URL}/key"
    r = requests.get(url, headers=headers, timeout=30)

    if r.status_code == 401:
        raise PermissionError("401 Unauthorized: API key salah, expired, atau tidak aktif.")
    if r.status_code == 403:
        raise PermissionError("403 Forbidden: API key tidak punya izin untuk endpoint ini.")

    r.raise_for_status()
    return r.json()["data"]

info = get_openrouter_key_info()

print("=== INFO API KEY OPENROUTER ===")
print("Label key          :", info.get("label"))
print("Free tier?         :", info.get("is_free_tier"))
print("Management key?    :", info.get("is_management_key"))
print("Expired at         :", info.get("expires_at"))

print("\n=== LIMIT DOLAR KEY ===")
print("Limit total key    :", money(info.get("limit")))
print("Sisa limit         :", money(info.get("limit_remaining")))
print("Reset limit        :", info.get("limit_reset") or "Tidak reset / tidak disetel")

print("\n=== PEMAKAIAN KEY ===")
print("Usage total        :", money(info.get("usage")))
print("Usage hari ini     :", money(info.get("usage_daily")))
print("Usage minggu ini   :", money(info.get("usage_weekly")))
print("Usage bulan ini    :", money(info.get("usage_monthly")))

print("\n=== BYOK USAGE, JIKA ADA ===")
print("BYOK usage total   :", money(info.get("byok_usage")))
print("BYOK hari ini      :", money(info.get("byok_usage_daily")))
print("BYOK minggu ini    :", money(info.get("byok_usage_weekly")))
print("BYOK bulan ini     :", money(info.get("byok_usage_monthly")))
print("BYOK masuk limit?  :", info.get("include_byok_in_limit"))

limit = info.get("limit")
remaining = info.get("limit_remaining")
usage = info.get("usage")

print("\n=== RINGKASAN ===")
if limit is None:
    print("Key ini tidak punya limit dolar khusus, atau limitnya tidak disetel.")
else:
    used_from_limit = limit - remaining if remaining is not None else usage
    percent = (used_from_limit / limit) * 100 if limit else 0

    print(f"Limit key          : {money(limit)}")
    print(f"Terpakai kira-kira : {money(used_from_limit)}")
    print(f"Sisa kira-kira     : {money(remaining)}")
    print(f"Persentase terpakai: {percent:.2f}%")

=== INFO API KEY OPENROUTER ===
Label key          : sk-or-v1-179...6ae
Free tier?         : False
Management key?    : False
Expired at         : None

=== LIMIT DOLAR KEY ===
Limit total key    : $35.000000
Sisa limit         : $10.769414
Reset limit        : Tidak reset / tidak disetel

=== PEMAKAIAN KEY ===
Usage total        : $24.230586
Usage hari ini     : $0.000000
Usage minggu ini   : $24.230586
Usage bulan ini    : $24.230586

=== BYOK USAGE, JIKA ADA ===
BYOK usage total   : $0.000000
BYOK hari ini      : $0.000000
BYOK minggu ini    : $0.000000
BYOK bulan ini     : $0.000000
BYOK masuk limit?  : False

=== RINGKASAN ===
Limit key          : $35.000000
Terpakai kira-kira : $24.230586
Sisa kira-kira     : $10.769414
Persentase terpakai: 69.23%
